# Кросс-парсинг PDF: Docling HTML + PyMuPDF

Демонстрация в Google Colab минимальной версии алгоритма кросс-парсинга нормативного PDF (Правила классификации и постройки морских судов РС).

**Пайплайн** (каждый этап — отдельный блок ниже):

| № | Блок | Движок | Что делает |
|---|------|--------|-----------|
| 1 | HTML Docling → JSON | Docling (HTML из git) | Постраничный разбор HTML: заголовки, абзацы, списки, таблицы, формулы; подмена «Formula not decoded» текстом из PDF |
| 2 | PyMuPDF-добор | PyMuPDF (PDF) | Добор пропущенного текста, детект формул (math-шрифты), заголовки титула по кеглю, восстановление bbox |
| 3 | Изображения | PyMuPDF (PDF) | Вырезание картинок и формул-растров, привязка image_key |
| 4 | Финальные чистки | Эвристики | Колонтитулы, склейка разорванных абзацев/формул, удаление дублей и пустых блоков |
| 5 | Качество | Метрики | Оценка confidence по страницам, сохранение JSON + HTML-визуализация |
| 6 | Контроль эталона | Сверка | Precision / Recall / F1, text coverage, расхождения |

**Источники данных** — всё берётся из git-репозитория: `data/pdf/` (PDF), `data/html/<doc>/` (постраничный HTML Docling + metadata), `data/etalon/` (эталоны), `app/` (алгоритмы).

**Секреты Colab** (значок «ключ» в левой панели):
- `GITHUB_REPO` — репозиторий (по умолчанию `neirokoder/cross_parsing`);
- `DOCLING_SERVE_URL` — адрес docling-serve (нужен только шагу docling-html; в демо HTML уже готов в git и serve не используется);
- `CROSS_PARSING_DATA` — каталог данных (необязательно).

Запуск: Runtime → Run all.

## 0. Установка зависимостей

Пакеты: `PyMuPDF` (движок PDF), `docling-core` (структуры Docling), `httpx` и `Pillow` (поддержка).

In [ ]:
!pip install -q PyMuPDF httpx "docling-core>=2.0" Pillow
print("deps OK")

## 1. Секреты Colab → переменные окружения (env)

Параметры читаются из секретов Colab (`userdata`) и пробрасываются в env **до** импорта `app.config`:
- `GITHUB_REPO` — репозиторий с кодом и данными;
- `DOCLING_SERVE_URL` — адрес docling-serve;
- `CROSS_PARSING_DATA` — каталог данных (опционально).

Если ноутбук запускается вне Colab, используется fallback на переменные окружения/дефолты.

In [ ]:
import os

try:
    from google.colab import userdata
except ImportError:
    class _Dummy:
        @staticmethod
        def get(key, default=None):
            return os.environ.get(key, default)
    userdata = _Dummy()

os.environ["GITHUB_REPO"] = userdata.get("GITHUB_REPO") or "neirokoder/cross_parsing"
os.environ["DOCLING_SERVE_URL"] = userdata.get("DOCLING_SERVE_URL") or "http://localhost:5001"
if userdata.get("CROSS_PARSING_DATA"):
    os.environ["CROSS_PARSING_DATA"] = userdata.get("CROSS_PARSING_DATA")

print("GITHUB_REPO =", os.environ["GITHUB_REPO"])
print("DOCLING_SERVE_URL =", os.environ["DOCLING_SERVE_URL"])

## 2. Код и данные из git

Клонируется репозиторий: `app/` (алгоритмы), `data/pdf/` (PDF-документы), `data/html/<doc>/` (постраничный HTML Docling + metadata), `data/etalon/` (эталоны).

In [ ]:
import subprocess
import sys
from pathlib import Path

BASE = Path("/content/cross_parsing")
if not (BASE / "app").exists():
    repo_url = f"https://github.com/{os.environ['GITHUB_REPO']}.git"
    r = subprocess.run(["git", "clone", "--depth", "1", repo_url, str(BASE)],
                       capture_output=True, text=True)
    print("clone OK" if r.returncode == 0 else (r.stdout + r.stderr).strip().splitlines()[-1])

sys.path.insert(0, str(BASE))
print("repo:", BASE)
print("docs:", sorted(p.name for p in (BASE / "data" / "html").iterdir() if p.is_dir()))

## 3. Демонстрационные документы

Два калиброванных документа (PDF + HTML Docling + эталон — всё в git). Для каждого заданы пути к PDF, каталогу HTML и эталону.

In [ ]:
DOCS = {
    "2-020101-174-5": {
        "pdf": BASE / "data" / "pdf" / "2-020101-174-5.pdf",
        "html_dir": BASE / "data" / "html" / "2-020101-174-5",
        "etalon": BASE / "data" / "etalon" / "2-020101-174-5_etalon_full.json",
    },
    "2-020101-174-12": {
        "pdf": BASE / "data" / "pdf" / "2-020101-174-12.pdf",
        "html_dir": BASE / "data" / "html" / "2-020101-174-12",
        "etalon": BASE / "data" / "etalon" / "2-020101-174-12_etalon_full.json",
    },
}
assert all(p.exists() for d in DOCS.values() for p in d.values()), "данные не найдены — проверь клон репозитория"
for name, p in DOCS.items():
    pages = len(list(p["html_dir"].glob("page_*.html")))
    print(f"{name}: html={pages} стр., pdf={p['pdf'].name}, etalon={p['etalon'].name}")

## Блок 1. HTML Docling → JSON (структура документа)

**Алгоритм** (`app/algorithm/html_to_json.py` + `pdf_extract.replace_not_decoded_formulas`):

1. **Загрузка** постраничного HTML (`page_XXXX.html`) и `metadata.json` (размеры страниц, bbox недекодированных формул).
2. **Подмена «Formula not decoded»**: Docling в режиме без OCR не декодирует формулы — по bbox из metadata текст формулы восстанавливается из PDF (PyMuPDF).
3. **Разбор HTML → блоки** (`html_to_document_json`): каждая страница превращается в блоки (heading/paragraph/list/table/image/formula) в формате opendataloader (`content.document.block[]`); нормализация ключей (у таблиц — `columns` отдельно, список — `items[]`), разрезы склеенных Docling'ом параграфов по маркерам пунктов (`_split_merged_paragraphs`), списки (`_split_lists`), формула+пояснение «, где …» (R1), условие «X = 0, если …» (`_split_zero_if`), дедупликация.

Результат — JSON структуры документа только по HTML Docling (ещё без PDF-добора).

In [ ]:
from app.algorithm.cross_parser import load_page_htmls
from app.algorithm.html_to_json import html_to_document_json
from app.algorithm import pdf_extract

results = {}
for doc, paths in DOCS.items():
    page_htmls, metadata = load_page_htmls(paths["html_dir"])

    formula_bboxes = (metadata or {}).get("formula_bboxes", {}) or {}
    formula_bboxes = {int(p): [list(map(float, b)) for b in bb] for p, bb in formula_bboxes.items()}
    page_htmls = pdf_extract.replace_not_decoded_formulas(page_htmls, str(paths["pdf"]), formula_bboxes)

    json_result = html_to_document_json(page_htmls, paths["pdf"].name)
    if metadata and metadata.get("pages"):
        json_result["content"]["document"]["pages"] = metadata["pages"]

    blocks = json_result["content"]["document"]["block"]
    results[doc] = {"json_result": json_result, "metadata": metadata}
    print(f"[{doc}] pages={len(page_htmls)}, blocks={len(blocks)}")

## Блок 2. PyMuPDF-добор: текст, формулы, заголовки титула, bbox

**Алгоритм** (`app/algorithm/pdf_extract.py`):

1. **Добор блоков из PDF** (`enrich_blocks_from_pdf`): текст PDF, пропущенный Docling'ом, добавляется в JSON (помечается `_enriched`); формулы детектируются по math-шрифтам и картинкам-растрам (`_line_math_share`).
2. **Заголовки титула** (`fix_title_page_headings`): Docling «уплощает» титул в h2 — уровни заголовков восстанавливаются по кеглю шрифта PDF (48→level 1, 24→level 2, 20/16→level 3), логотип издателя «РОССИЙСКИЙ МОРСКОЙ РЕГИСТР СУДОХОДСТВА» переводится в paragraph.
3. **bbox из PDF** (`restore_bboxes`): позиции блоков проставляются по совпадению текста с координатами из PDF.
4. **bbox из Docling** (`restore_docling_bboxes`): провенансы `docling_document.json` (BOTTOMLEFT → TOPLEFT через высоту страницы); заполняются только блоки без bbox; если файла нет — шаг пропускается (для 174-12).

In [ ]:
for doc, r in results.items():
    jr = r["json_result"]
    pdf = DOCS[doc]["pdf"]
    bbox_map = pdf_extract.enrich_blocks_from_pdf(jr, str(pdf))
    pdf_extract.fix_title_page_headings(jr, str(pdf))
    from_pdf = pdf_extract.restore_bboxes(jr, str(pdf), bbox_map)
    from_doc = pdf_extract.restore_docling_bboxes(jr, str(DOCS[doc]["html_dir"] / "docling_document.json"))
    r["bbox_map"] = bbox_map
    n = len(jr["content"]["document"]["block"])
    print(f"[{doc}] blocks={n}, bbox: pdf={from_pdf}, docling={from_doc}")

## Блок 3. Изображения и формулы-растры из PDF

**Алгоритм** (`pdf_extract`):

1. `save_images_from_pdf` — вырезает растровые изображения страниц (PNG в `data/images/<doc>/`).
2. `update_image_keys` — привязывает сохранённые файлы к image-блокам по геометрии.
3. `inject_missing_image_blocks` — добавляет image-блоки, пропущенные Docling'ом.
4. `extract_formula_images` — вырезает формулы-картинки; IoU-отсев по bbox таблиц-растров (чтобы таблицу не задвоить формулой).

In [ ]:
for doc, r in results.items():
    jr = r["json_result"]
    images_dir = BASE / "data" / "images" / doc
    images_dir.mkdir(parents=True, exist_ok=True)
    pdf_images = pdf_extract.save_images_from_pdf(str(DOCS[doc]["pdf"]), str(images_dir))
    pdf_extract.update_image_keys(jr, str(images_dir), pdf_images)
    pdf_extract.inject_missing_image_blocks(jr, str(images_dir), pdf_images)
    pdf_extract.extract_formula_images(jr, str(DOCS[doc]["pdf"]), str(images_dir), r["bbox_map"])
    r["images_dir"] = images_dir
    print(f"[{doc}] images saved: {len(pdf_images)}")

## Блок 4. Финальные чистки (`_final_cleanup`)

Эвристики поверх результата кросс-парсинга (`app/algorithm/cross_parser.py`), устраняют артефакты Docling и PDF-добора:

- Заголовок-одиночка в предложном регистре («Санкт-Петербург») → paragraph;
- Колонтитулы «Правила классификации и постройки морских судов…» — удаление;
- Пояснение «где …» после формулы → paragraph; сборка разорванного «где Np — …»;
- Склейка разорванных Docling-абзацев (фрагмент начинается со строчной буквы и prev не оканчивается на `.?!…:;`); маркерные фрагменты «10.1.13 …» со склейкой маркера;
- Склейка фрагментов формул (по завершающей пунктуации и операторам продолжения строки), многострочные формулы — слияние по геометрии bbox, хвосты условий «для ρ ≤1400 кг/м3 …»;
- Восстановление «X = 0, если …» (Docling переставляет фрагменты: голова формулы и хвост фразы склеены, середина вырезана);
- Удаление пустых формул/списков и дублей по нормализованному тексту страницы (точные, подстроковые, нечёткие, префиксные, формулы);
- Подписи «Рис. N.N.N-N» — переносятся сразу после своего изображения.

In [ ]:
from app.algorithm.cross_parser import _final_cleanup

for doc, r in results.items():
    jr = r["json_result"]
    before = len(jr["content"]["document"]["block"])
    _final_cleanup(jr)
    after = len(jr["content"]["document"]["block"])
    print(f"[{doc}] cleanup: {before} -> {after} blocks")

## Блок 5. Качество страниц и сохранение результата

**Алгоритм** (`_assess_quality` + `app/algorithm/quality_metrics.py`): для каждой страницы по блокам считаются метрики (типы блоков, плотность текста, аномалии) и confidence 0..1; страницы с низким confidence помечаются `low_confidence`. Общий confidence документа — среднее по страницам.

**Сохранение**: `data/output/<doc>.json` (raw_ocr_v4; пути картинок — относительные от корня проекта) + `data/output/<doc>.html` — единая HTML-визуализация по типам блоков с навигацией по страницам (`json_to_html`).

In [ ]:
import json
import os
from app.algorithm.cross_parser import _assess_quality
from app.algorithm.json_to_html import json_to_html

for doc, r in results.items():
    jr = r["json_result"]
    quality = _assess_quality(jr)
    jr["content"]["quality"].update(quality)

    out_json = BASE / "data" / "output" / f"{doc}.json"
    out_json.parent.mkdir(parents=True, exist_ok=True)
    for blk in jr["content"]["document"]["block"]:
        tp = blk.get("_temp_path")
        if tp:
            blk["_temp_path"] = os.path.relpath(tp, BASE)
    out_json.write_text(json.dumps(jr, ensure_ascii=False, indent=2), encoding="utf-8")
    json_to_html(jr, out_json.with_suffix(".html"))
    r["out_json"] = out_json

    q = jr["content"]["quality"]
    n = len(jr["content"]["document"]["block"])
    print(f"[{doc}] confidence={q['confidence']} pages={q['pages_processed']} blocks={n}")

## Блок 6. Контроль эталона (Precision / Recall / F1)

**Алгоритм** (`app/etalon/compare.py`):

- **Жадный матч по странице**: для каждого блока эталона ищется лучший блок результата на той же странице по сходству текста (SequenceMatcher, порог 0.6); точные совпадения приоритетны.
- **Матч только между блоками одного типа** — иначе «совпадение из разных участков JSON» засчитывается как `type_mismatch`, а не как совпадение.
- **Image без текста** сопоставляется по `image_key`.
- **Метрики**: Precision / Recall / F1 по блокам; `text_coverage` — доля символов эталона, покрытая текстом совпавших пар; `structure_mismatch` — расхождение формы JSON у совпавших пар (columns/rows, items, level, image_key).

Эталон — ручная семантическая разметка документа (лежит в `data/etalon/`). Цель работы пайплайна — F1 ≈ 1.0.

In [ ]:
from app.etalon.compare import compare
from app.etalon.report import render_markdown
from IPython.display import Markdown

summary_rows = []
for doc, r in results.items():
    report = compare(str(DOCS[doc]["etalon"]), str(r["out_json"]))
    r["report"] = report
    s = report["summary"]
    summary_rows.append((doc, s["etalon_blocks"], s["result_blocks"], s["matched"],
                         s["precision"], s["recall"], s["f1"], s["text_coverage"],
                         s["type_mismatch"], s.get("structure_mismatch", 0)))
    display(Markdown(f"### {doc}\n" + render_markdown(report)))

print("=== Сводка ===")
print(f"{'документ':<22}{'эталон':>7}{'рез-т':>8}{'совпало':>8}{'P':>7}{'R':>7}{'F1':>7}{'cov':>7}{'тип':>5}{'структ':>8}")
for row in summary_rows:
    print(f"{row[0]:<22}{row[1]:>7}{row[2]:>8}{row[3]:>8}{row[4]:>7}{row[5]:>7}{row[6]:>7}{row[7]:>7}{row[8]:>5}{row[9]:>8}")

## Блок 7. Просмотр результата

Единая HTML-визуализация результата парсинга (все страницы, навигация сверху) встроена в ноутбук.

In [ ]:
from IPython.display import IFrame

for doc, r in results.items():
    print("====" , doc, "====")
    display(IFrame(src=str(r["out_json"].with_suffix(".html")), width="100%", height=600))